In [ ]:
# 2026-06-03
from apis_core.apis_metainfo.models import Uri, Collection
from csae_pyutils import gsheet_to_df

from normdata.utils import import_from_normdata
from tqdm import tqdm
from time import sleep

In [ ]:
col = Collection.objects.filter(name__startswith="ÖBL nach").first()
domain = "oebl"


In [ ]:
df = gsheet_to_df("1TnVJu6V4jcRDiL5wJ_GlEG19pr7hZaZcic9B09OswC8")

In [ ]:
df.head()

In [ ]:
filtered = df[df["doi"].notna()]
print(len(df), len(filtered))

In [ ]:
for wikidata_id, ndf in tqdm(filtered.groupby('person')):
    doi = f"https://doi.org/{ndf.iloc[0]['doi']}"
    new_uri, created = Uri.objects.get_or_create(uri=doi, domain=domain)
    if created:
        sleep(1)
    entity = import_from_normdata(wikidata_id, "person")
    new_uri.entity = entity
    new_uri.save()
    entity.collection.add(col)
